# Day 3 — Forecast a series and classify text

All data are synthetic. Run from top to bottom. Work in pairs and pause after each result to explain its meaning. Use TEACHING_GUIDE.md and TASK_CARDS.md for timing. Optional sections are marked. Numerical outputs are examples, not evidence about real operations.

## 1. Build an ordered series

Daily synthetic demand contains a trend and a seven-day pattern. Hold out the final 28 days before fitting or choosing models. Do not shuffle future and past.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
rng=np.random.default_rng(33)
t=np.arange(210)
series=pd.Series(120+.15*t+18*np.sin(2*np.pi*t/7)+rng.normal(0,5,len(t)),index=pd.date_range('2026-01-01',periods=len(t),freq='D'),name='demand')
train,test=series.iloc[:-28],series.iloc[-28:]
print('Training:',train.index.min().date(),train.index.max().date())
print('Final test:',test.index.min().date(),test.index.max().date())
train.plot(figsize=(8,3),title='Training demand only');plt.ylabel('Units per day');plt.tight_layout();plt.show()

## 2. Trend, seasonality and baselines

A seven-day trailing mean smooths the series; it is not a complete statistical decomposition. Lag-7 correlation can reflect seasonality and trend together. Last-value and weekly-repeat forecasts use only past observations. The two baseline rules are predeclared, not selected on the final test.

In [ ]:
rolling=train.rolling(7).mean()
print('Lag-7 correlation:',round(train.autocorr(7),3))
fig,ax=plt.subplots(figsize=(8,3));train.plot(ax=ax,label='Observed');rolling.plot(ax=ax,label='Trailing 7-day mean');ax.legend();ax.set_ylabel('Demand units');plt.tight_layout();plt.show()
last=np.repeat(train.iloc[-1],len(test))
weekly=np.resize(train.iloc[-7:].to_numpy(),len(test))
print('Last-value test MAE:',round(mean_absolute_error(test,last),2))
print('Weekly-repeat test MAE:',round(mean_absolute_error(test,weekly),2))

## 3. ARIMA and decomposition — statsmodels required

ARIMA(p,d,q) combines autoregressive lags, differencing and moving-average error terms. Here (1,1,1) is a fixed illustration, not a tuned optimum. It does not explicitly model the known weekly seasonality. A model may lose to a seasonal baseline. Decomposition is fitted only to training data; its residuals are not forecast validation.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
parts=seasonal_decompose(train,model='additive',period=7)
parts.plot();plt.tight_layout();plt.show()
fit=ARIMA(train,order=(1,1,1)).fit()
forecast=fit.get_forecast(steps=len(test))
pred=forecast.predicted_mean
interval=forecast.conf_int()
print('ARIMA final-test MAE:',round(mean_absolute_error(test,pred),2))
print('Optimiser convergence:',fit.mle_retvals.get('converged'))
fig,ax=plt.subplots(figsize=(8,3));ax.plot(test.index,test.to_numpy(),label='Actual');ax.plot(pred.index,pred.to_numpy(),label='ARIMA')
ax.plot(test.index,weekly,label='Weekly baseline')
ax.fill_between(test.index,interval.iloc[:,0],interval.iloc[:,1],alpha=.2,label='Model 95% interval')
ax.set_ylabel('Demand units');ax.legend();plt.tight_layout();plt.show()

## 4. Text classification — a separate problem

Classify short English service messages into delivery or billing. These deliberately small authored examples teach a pipeline, not deployment accuracy or Arabic NLP. Training and test messages are listed separately; vocabulary is learned only from training text.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
texts=['delivery arrived late','where is my parcel','courier missed the address','shipment is delayed','package arrived damaged','tracking shows no delivery','driver could not find my building','my order has not arrived',
'invoice has an incorrect amount','payment was charged twice','please send my receipt','refund is missing','billing address is incorrect','card payment was rejected','the invoice includes extra charges','I need a payment refund']
labels=['delivery']*8+['billing']*8
test_texts=['parcel delivery is delayed','courier delivered a damaged package','tracking says my shipment arrived','driver missed my address','invoice charged twice','please refund this payment','receipt amount is incorrect','card charge is missing']
test_labels=['delivery']*4+['billing']*4
text_model=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(C=2,max_iter=1000))
text_model.fit(texts,labels)
text_pred=text_model.predict(test_texts)
print(pd.DataFrame({'message':test_texts,'actual':test_labels,'predicted':text_pred}).to_string(index=False))
print(classification_report(test_labels,text_pred,zero_division=0))
print('Ambiguous message prediction:',text_model.predict(['my order payment is late'])[0])
print('A forced label is not proof of reliable understanding.')